# Sandbox de Exploração - Modelo M3 (Expected Gain)
Objetivo: Testar a previsão do Expected Gain, avaliando várias técnicas de normalização e modelos alternativos antes do XGBoost final.

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings('ignore')

sys.path.append(os.path.abspath('../scripts'))
from prev_gain import load_data, build_feature_matrix, get_feature_cols

_PG_HOST = os.environ.get("PG_HOST", "localhost")
_PG_PORT = os.environ.get("PG_PORT", "5432")
URL_DW = f"postgresql+psycopg2://ae_user:ae_pass_2026@{_PG_HOST}:{_PG_PORT}/auto_escala"

engine = create_engine(URL_DW)
print("Engine conectada com sucesso!")

In [ ]:
# Carregar dados usando os módulos do script de produção
with engine.connect() as conn:
    df_feat, df_tgt, df_dem = load_data(conn, 'auto_escala_dw')
    
df_m3 = build_feature_matrix(df_feat, df_tgt, df_dem)
feature_cols = get_feature_cols(df_m3)

print(f"Tamanho do dataset: {len(df_m3)}")
display(df_m3.head())

## 1. Split Temporal e Normalização
Vamos separar os dados temporalmente (últimos meses para teste). Como temos variáveis numéricas diversas, vamos avaliar o impacto de não as normalizar vs aplicar StandardScaler.

In [ ]:
# Encontrar o último mês disponível
max_ano = df_m3["ano"].max()
max_mes = df_m3[df_m3["ano"] == max_ano]["mes"].max()

treino = df_m3[(df_m3["ano"] < max_ano) | ((df_m3["ano"] == max_ano) & (df_m3["mes"] < max_mes))]
teste = df_m3[(df_m3["ano"] == max_ano) & (df_m3["mes"] == max_mes)]

X_train, y_train = treino[feature_cols], treino["target_next"]
X_test, y_test = teste[feature_cols], teste["target_next"]

# Com StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Treino: {len(X_train)} amostras | Teste: {len(X_test)} amostras")

## 2. Testes com Modelos Base (Ridge e Random Forest)
Avaliar o impacto da normalização na Regressão Ridge e explorar árvores não normais.

In [ ]:
# Ridge Sem Escalar
ridge_raw = Ridge()
ridge_raw.fit(X_train, y_train)
pred_ridge_raw = ridge_raw.predict(X_test)

# Ridge Com StandardScaler
ridge_scaled = Ridge()
ridge_scaled.fit(X_train_scaled, y_train)
pred_ridge_scaled = ridge_scaled.predict(X_test_scaled)

# Random Forest (não precisa de escalar)
rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

print(f"MAE Ridge (Raw): {mean_absolute_error(y_test, pred_ridge_raw):.4f}")
print(f"MAE Ridge (Scaled): {mean_absolute_error(y_test, pred_ridge_scaled):.4f}")
print(f"MAE Random Forest: {mean_absolute_error(y_test, pred_rf):.4f}")

## 3. Modelo Final Escolhido: XGBoost
Optámos pelo XGBoost por conseguir lidar excecionalmente bem com valores nulos (que temos muitos em variáveis que surgem em algumas marcas e não outras) e não requerer normalização do dataset.

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(X_train, y_train)
pred_xgb = xgb_model.predict(X_test)

print(f"MAE XGBoost Final: {mean_absolute_error(y_test, pred_xgb):.4f}")

# Plot Importância das Features
importances = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values(ascending=True)

plt.figure(figsize=(10, 8))
importances.tail(15).plot(kind='barh')
plt.title("Top 15 Features mais importantes para XGBoost (Expected Gain)")
plt.tight_layout()
plt.show()